In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    TrainingArguments, 
    Trainer
)
from torch.utils.data import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import random

In [ ]:
# Set all random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Custom Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.15, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss

# Enhanced model with additional features (now 2 features instead of 3)
class RobertaWithFeatures(nn.Module):
    def __init__(self, model_name, num_labels=2, extra_feat_dim=2):
        super(RobertaWithFeatures, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + extra_feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, extra_feats, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]  # [CLS] token
        combined = torch.cat((cls_output, extra_feats), dim=1)
        logits = self.classifier(combined)

        if labels is not None:
            return {'logits': logits, 'labels': labels}
        return {'logits': logits}

# Custom dataset with additional features
class RelevanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        topic = str(row['MAIN'])
        comment = str(row['comment_body'])
        label = int(row['relevance'])

        encoding = self.tokenizer(
            topic, 
            comment, 
            truncation=True, 
            padding='max_length',
            max_length=self.max_length, 
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'extra_feats': torch.tensor([
                row['cos_sim'],
                row['bert_score']
                # Removed comment_score
            ], dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Custom trainer with focal loss
class CustomTrainer(Trainer):
    def __init__(self, *args, focal_loss=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = focal_loss

    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"]
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Compute metrics function
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    accu = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    return {
        "accuracy": accu,
        "f1": f1,
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "comb": accu + f1,
    }

def main():
    # Load and preprocess data
    df = pd.read_csv('/kaggle/input/task2-features/qna_train_Features.csv')
    
    # Split data into train and validation sets
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    
    # Initialize tokenizer and model
    model_name = "roberta-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    MAX_LENGTH = 512
    
    # Create datasets
    train_dataset = RelevanceDataset(train_df, tokenizer, MAX_LENGTH)
    val_dataset = RelevanceDataset(val_df, tokenizer, MAX_LENGTH)
    
    # Initialize model
    model = RobertaWithFeatures(model_name=model_name)
    
    # Define loss
    focal_loss = FocalLoss(alpha=0.15, gamma=2.0)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir="./roberta-results",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        seed=42
    )
    
    # Initialize trainer
    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        focal_loss=focal_loss,
    )
    
    # Train the model
    trainer.train()
    
    # Save the model
    trainer.save_model("./final_model")

if __name__ == "__main__":
    main()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Comb
1,0.012900,0.013722,0.880492,0.443763,0.632653,0.341732,1.324255
2,0.013900,0.012833,0.893014,0.388959,0.956790,0.244094,1.281973
